# Notebook 1: Short-Term Memory — Agent State + Checkpointer

### What it does (simplest version)
Remembers everything said **in the current conversation**, so follow-ups like
*"yes"*, *"make it shorter"*, *"what about the second one?"* actually work.

### Human analogy
Your memory **during one phone call**. You remember everything the other person said
since the call started — but the moment you hang up and call someone else, it resets.

### The 3 pieces (learn these names)
| Piece | What it is | One-line job |
|---|---|---|
| **Agent State** | The agent's data — here, just a list of `messages` | Where the memory *lives* |
| **Checkpointer** | A saver attached when we `compile()` the graph | *Saves* the state after every turn |
| **thread_id** | A name you give each conversation | *Which* conversation to load |

The flow each turn:
```
your new message  →  checkpointer loads saved state (old messages)
                  →  LLM sees everything  →  reply
                  →  checkpointer saves the updated state
```

### Why it matters
Without a checkpointer, every call is a stranger call (we proved this in Notebook 0).

### What it connects with later
- This notebook = the **foundation**. Notebooks 2 and 3 only change *what part of the state* the LLM sees.
- Its weakness: memory dies with the `thread_id`. Notebooks 4–6 fix that with the **Store** (long-term memory).

## Step 1 — Setup
Same as Notebook 0: load the key, create the model.

In [ ]:
from dotenv import load_dotenv
load_dotenv()

from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage
from langgraph.graph import StateGraph, MessagesState, START, END
from langgraph.checkpoint.memory import InMemorySaver

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
print("Setup done.")

## Step 2 — Build the agent (this is the "Agent State" part)

- `MessagesState` = the agent's state. It is literally a list of messages that grows each turn.
- A **node** is one step of work. Our node calls the LLM with the *whole* state and returns the reply,
  which gets **appended** to `messages` automatically.

In [ ]:
def chatbot(state: MessagesState):
    # state["messages"] already contains ALL past messages of this thread
    # (restored by the checkpointer before this node runs)
    response = llm.invoke(state["messages"])
    return {"messages": [response]}   # appended to state automatically

builder = StateGraph(MessagesState)
builder.add_node("chatbot", chatbot)
builder.add_edge(START, "chatbot")
builder.add_edge("chatbot", END)

# ★★★ THE memory line ★★★
# Compiling WITH a checkpointer = "save the state after every turn"
graph = builder.compile(checkpointer=InMemorySaver())
print("Graph compiled with memory.")

## Step 3 — Give the conversation a name: `thread_id`

Think of `thread_id` as a **file name** for the conversation.
Same `thread_id` → the checkpointer reloads the same history.
Different `thread_id` → a brand-new empty history.

In [ ]:
# "class-demo-1" = the name of this conversation
config = {"configurable": {"thread_id": "class-demo-1"}}

# Turn 1
r1 = graph.invoke(
    {"messages": [HumanMessage(content="Hi, my name is Rahul and I love biryani.")]},
    config,
)
print("AI:", r1["messages"][-1].content)

## Step 4 — The magic moment: turn 2

Look closely — we send **only the new message**. We never resend turn 1.
The checkpointer loads turn 1 from the saved state and glues it together for us.

In [ ]:
# Turn 2 — only the NEW message is sent
r2 = graph.invoke(
    {"messages": [HumanMessage(content="What is my name and what food do I love?")]},
    config,   # same thread_id -> old messages are loaded
)
print("AI:", r2["messages"][-1].content)

# Let's also look at the FULL saved state — this is the agent's memory:
print("Messages stored in state:")
for m in r2["messages"]:
    print(f"  {m.type}: {m.content[:60]}")

## Step 5 — Proof that memory is per-thread

A **different** `thread_id` = a stranger. This is the "Current Thread" idea from the diagram:
short-term memory belongs to ONE conversation only.

In [ ]:
config2 = {"configurable": {"thread_id": "class-demo-2"}}   # different conversation!

r3 = graph.invoke(
    {"messages": [HumanMessage(content="What is my name?")]},
    config2,
)
print("AI (new thread):", r3["messages"][-1].content)
# -> It does NOT know. New thread = empty memory.

## Try it yourself

**Experiment:** remove the checkpointer and watch memory die.

```python
graph_no_memory = builder.compile()          # <- no checkpointer!
config = {"configurable": {"thread_id": "x"}}
graph_no_memory.invoke({"messages": [HumanMessage("My name is Rahul.")]}, config)
out = graph_no_memory.invoke({"messages": [HumanMessage("What is my name?")]}, config)
print(out["messages"][-1].content)           # -> doesn't know
```

**Questions to answer in class:**
1. What are the 3 pieces that make short-term memory work?
2. If two users share one `thread_id`, what happens? (Try it!)
3. `InMemorySaver` stores memory in RAM. What happens when the script ends?
   (That's why production apps use a database checkpointer — the code stays identical.)

**Next notebook:** full history forever gets expensive. Notebook 2 = the *window* trick.